# Classificador de Integridade do Universo — Exclusão Estática

**Apêndice — TCC (otimização de carteiras, B3).**

Classifica cada ticker do universo final em **MANTER** ou **EXCLUIR**, por um critério **objetivo e uniforme**, usando o **COTAHIST tratado** (preço nominal bruto e flag oficial de situação especial). Não há máscara mensal nem reengenharia do pipeline: a saída é uma **lista de exclusão** para você revisar e aplicar uma única vez.

**Regra de exclusão** (decidida com o orientando):
- **Penny:** o mínimo anual de `PRECO_UNIT` cai **abaixo de R$ 1 em qualquer ano** do período 2010–2025; **ou**
- **Situação especial:** `FLAG_SITUACAO=True` em **qualquer pregão** (RJ, concordata, RAET, intervenção — `CODBDI` 06–11).

**Linhagem:** trocas de código societárias (ex.: `FICT3`←`ATOM3`, `GOLL54`←`GOLL4`) são avaliadas **em conjunto** via um mini-mapa explícito — o ticker atual herda o histórico do predecessor. O notebook também **sugere** predecessores por continuidade de datas, para você revisar.

> Justificativa para o Cap. 3: o preço **ajustado** da Economática é deformado por eventos futuros (ajuste retroativo de grupamento), logo impróprio para julgar penny stock; usa-se o **preço bruto** do COTAHIST. A distorção da cotação é *evidência corroborante*, não o critério.

In [3]:
from pathlib import Path
import pandas as pd
import numpy as np



# ============================ CONFIGURAÇÃO ============================
PASTA_COTAHIST = Path(r"C:\VSCodeWorkspace\1_TCC_Final\data\B3_COTAHIST_Tratatos")
ARQ_UNIVERSO   = Path(r"C:\VSCodeWorkspace\1_TCC_Final\data\Painel_Dados\tickers_finais.csv")
PASTA_SAIDA    = Path(r"C:\VSCodeWorkspace\1_TCC_Final\data")

PISO_PRECO       = 1.00          # piso de penny stock (R$) sobre PRECO_UNIT
ANO_INI, ANO_FIM = 2010, 2025    # janela de avaliação da regra

# Mini-mapa de linhagem: {codigo_atual: [predecessores]} — edite conforme a sugestão da Seção 3
MAPA_LINHAGEM = {
    "FICT3":  ["ATOM3"],   # Fictor Alimentos <- Atom Participações
    "GOLL54": ["GOLL4"],   # Gol PN pós-Chapter 11
}

DETECTAR_LINHAGEM = True         # sugere predecessores por continuidade de datas (revisão manual)
CONTINUIDADE_DIAS = 7            # janela (dias) para considerar "encaixe" de séries
# =====================================================================

## 1. Leitura do universo e dos resumos por ticker

Lê **uma vez** cada arquivo do COTAHIST tratado (preferindo `.parquet`) e guarda só um **resumo leve** por ticker: faixa de datas, mínimo e mediana de `PRECO_UNIT` por ano, e anos com situação especial. Memória baixa, mesmo com centenas de tickers.

In [4]:
def carregar_universo(arq):
    u = pd.read_csv(arq)
    col = 'ticker' if 'ticker' in u.columns else u.columns[0]
    return u[col].astype(str).str.strip().tolist()

def ler_ticker(pasta_ticker: Path):
    """Lê o arquivo de um ticker (parquet preferido), só as colunas necessárias."""
    nome = pasta_ticker.name
    pq, csv = pasta_ticker / f"{nome}.parquet", pasta_ticker / f"{nome}.csv"
    cols = ['DATA_PREGAO', 'ANO', 'PRECO_UNIT', 'FLAG_SITUACAO', 'CODBDI']
    if pq.exists():
        d = pd.read_parquet(pq, columns=[c for c in cols if True])
    elif csv.exists():
        d = pd.read_csv(csv, usecols=lambda c: c in cols, parse_dates=['DATA_PREGAO'])
    else:
        return None
    d['ANO'] = pd.to_numeric(d['ANO'], errors='coerce').astype('Int64')
    d['PRECO_UNIT'] = pd.to_numeric(d['PRECO_UNIT'], errors='coerce')
    d['FLAG_SITUACAO'] = d['FLAG_SITUACAO'].astype(bool)
    return d

def resumir(d: pd.DataFrame) -> dict:
    dd = d[d['ANO'].between(ANO_INI, ANO_FIM)]
    g = dd.groupby('ANO')['PRECO_UNIT']
    sit = dd[dd['FLAG_SITUACAO']]
    return {
        'primeiro': dd['DATA_PREGAO'].min(), 'ultimo': dd['DATA_PREGAO'].max(), 'n': len(dd),
        'min_por_ano': {int(a): float(v) for a, v in g.min().items() if pd.notna(v)},
        'med_por_ano': {int(a): float(v) for a, v in g.median().items() if pd.notna(v)},
        'anos_sit': sorted({int(a) for a in sit['ANO'].dropna().unique()}),
        'codbdi_sit': sorted(sit['CODBDI'].dropna().unique().tolist()),
    }

if not PASTA_COTAHIST.exists():
    raise FileNotFoundError(f"Pasta COTAHIST não encontrada: {PASTA_COTAHIST}")

universo = carregar_universo(ARQ_UNIVERSO)
print(f"Universo final: {len(universo)} tickers.")

pastas = [p for p in PASTA_COTAHIST.iterdir() if p.is_dir()]
print(f"Lendo resumos de {len(pastas)} ticker(s) do COTAHIST tratado...")
RESUMO = {}
for i, p in enumerate(pastas, 1):
    d = ler_ticker(p)
    if d is not None and len(d):
        RESUMO[p.name] = resumir(d)
    if i % 200 == 0:
        print(f"  {i}/{len(pastas)}")
print(f"Resumos prontos: {len(RESUMO)} tickers.")

Universo final: 122 tickers.
Lendo resumos de 893 ticker(s) do COTAHIST tratado...
  200/893
  400/893
  600/893
  800/893
Resumos prontos: 893 tickers.


## 2. (Opcional) Sugestão de predecessores por continuidade de datas

Para cada ticker do universo que **não** começa no início da janela, procura outro código cuja série **termina poucos dias antes** — candidato a predecessor (renomeação/reorganização). **Revise** e adicione os confirmados ao `MAPA_LINHAGEM`. Candidatos que foram penny/RJ são marcados com ⚠ (alta prioridade).

In [5]:
if DETECTAR_LINHAGEM:
    fim_por_tk = {t: r['ultimo'] for t, r in RESUMO.items() if pd.notna(r['ultimo'])}
    achou = False
    print("Candidatos a predecessor (revise antes de confiar):\n")
    for t in universo:
        r = RESUMO.get(t)
        if r is None or pd.isna(r['primeiro']):
            continue
        ini = r['primeiro']
        if ini <= pd.Timestamp(ANO_INI, 1, 31):   # já começa no início -> sem predecessor relevante
            continue
        for cand, fim in fim_por_tk.items():
            if cand == t:
                continue
            delta = (ini - fim).days
            if 1 <= delta <= CONTINUIDADE_DIAS:
                rc = RESUMO[cand]
                era_penny = any(v < PISO_PRECO for v in rc['min_por_ano'].values())
                marca = " ⚠ (foi penny/RJ)" if (era_penny or rc['anos_sit']) else ""
                ja = " [já no mapa]" if cand in MAPA_LINHAGEM.get(t, []) else ""
                print(f"  {t}  <-  {cand}  (gap {delta}d; {cand} até {fim.date()}){marca}{ja}")
                achou = True
    if not achou:
        print("  Nenhum candidato por continuidade de datas.")
else:
    print("Detecção de linhagem desativada.")

Candidatos a predecessor (revise antes de confiar):

  ABEV3  <-  AMBV3  (gap 3d; AMBV3 até 2013-11-08)
  ABEV3  <-  AMBV4  (gap 3d; AMBV4 até 2013-11-08)
  ABEV3  <-  VINE3  (gap 7d; VINE3 até 2013-11-04)
  AMAR3  <-  MARI3  (gap 3d; MARI3 até 2010-06-25)
  AXIA3  <-  CPLE6  (gap 3d; CPLE6 até 2025-11-07)
  AXIA3  <-  ELET3  (gap 3d; ELET3 até 2025-11-07)
  AXIA3  <-  ELET6  (gap 3d; ELET6 até 2025-11-07)
  AXIA3  <-  EQPA6  (gap 5d; EQPA6 até 2025-11-05)
  AXIA3  <-  USIM6  (gap 3d; USIM6 até 2025-11-07)
  AXIA6  <-  CPLE6  (gap 3d; CPLE6 até 2025-11-07)
  AXIA6  <-  ELET3  (gap 3d; ELET3 até 2025-11-07)
  AXIA6  <-  ELET6  (gap 3d; ELET6 até 2025-11-07)
  AXIA6  <-  EQPA6  (gap 5d; EQPA6 até 2025-11-05)
  AXIA6  <-  USIM6  (gap 3d; USIM6 até 2025-11-07)
  B3SA3  <-  BVMF3  (gap 3d; BVMF3 até 2018-03-23)
  CLSC4  <-  CLSC6  (gap 1d; CLSC6 até 2012-01-19)
  CLSC4  <-  UOLL4  (gap 2d; UOLL4 até 2012-01-18)
  CSUD3  <-  CARD3  (gap 1d; CARD3 até 2022-09-14)
  CSUD3  <-  SHUL3  (gap 3d; 

## 3. Classificação MANTER / EXCLUIR

Combina o ticker com seus predecessores (`MAPA_LINHAGEM`): penny = mínimo anual abaixo do piso em **qualquer** código da linhagem; situação especial = flag em **qualquer** código. O motivo registra **qual código** disparou (marcando *via predecessor*). Tickers sem dado no COTAHIST ficam como **SEM DADOS** para revisão manual (nunca mantidos silenciosamente).

In [6]:
def combinar_min_por_ano(codigos):
    comb = {}
    for c in codigos:
        for a, v in RESUMO[c]['min_por_ano'].items():
            comb[a] = min(comb.get(a, v), v)
    return comb

linhas = []
for t in universo:
    codigos = [t] + MAPA_LINHAGEM.get(t, [])
    presentes = [c for c in codigos if c in RESUMO]
    if not presentes:
        linhas.append({'ticker': t, 'decisao': 'SEM DADOS', 'motivo': 'sem série no COTAHIST tratado',
                       'penny': pd.NA, 'situacao_especial': pd.NA, 'anos_penny': '', 'anos_situacao': '',
                       'min_preco_unit': pd.NA, 'codigos_linhagem': ';'.join(codigos),
                       'cobertura': False, 'primeiro_pregao': pd.NaT, 'ultimo_pregao': pd.NaT})
        continue

    # penny por código (para detalhar a origem) e anos de situação por código
    penny_por_cod = {c: sorted(a for a, v in RESUMO[c]['min_por_ano'].items() if v < PISO_PRECO) for c in presentes}
    penny_por_cod = {c: ay for c, ay in penny_por_cod.items() if ay}
    sit_por_cod   = {c: RESUMO[c]['anos_sit'] for c in presentes if RESUMO[c]['anos_sit']}

    penny = len(penny_por_cod) > 0
    situ  = len(sit_por_cod) > 0

    motivos = []
    if penny:
        det = ", ".join(f"{c} {ay}{' (via predecessor)' if c != t else ''}" for c, ay in penny_por_cod.items())
        motivos.append(f"penny: PRECO_UNIT < R${PISO_PRECO:.0f} em {det}")
    if situ:
        det = ", ".join(f"{c} {ay}{' (via predecessor)' if c != t else ''}" for c, ay in sit_por_cod.items())
        motivos.append(f"situacao especial em {det}")

    comb_min = combinar_min_por_ano(presentes)
    anos_penny_todos = sorted(a for a, v in comb_min.items() if v < PISO_PRECO)
    anos_sit_todos   = sorted({a for c in presentes for a in RESUMO[c]['anos_sit']})

    linhas.append({
        'ticker': t,
        'decisao': 'EXCLUIR' if motivos else 'MANTER',
        'motivo': "; ".join(motivos),
        'penny': penny, 'situacao_especial': situ,
        'anos_penny': ",".join(map(str, anos_penny_todos)),
        'anos_situacao': ",".join(map(str, anos_sit_todos)),
        'min_preco_unit': round(min(comb_min.values()), 4) if comb_min else pd.NA,
        'codigos_linhagem': ";".join(presentes),
        'cobertura': True,
        'primeiro_pregao': min(RESUMO[c]['primeiro'] for c in presentes),
        'ultimo_pregao':  max(RESUMO[c]['ultimo']  for c in presentes),
    })

classif = pd.DataFrame(linhas).sort_values(['decisao', 'ticker']).reset_index(drop=True)
print(classif['decisao'].value_counts().to_string())

decisao
MANTER       93
EXCLUIR      28
SEM DADOS     1


## 4. Saídas e resumo

Grava três arquivos: a classificação completa dos 122, a lista de exclusão (com motivo) e o universo já filtrado (`tickers_finais_v2.csv`).

In [7]:
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

classif.to_csv(PASTA_SAIDA / "classificacao_universo.csv", index=False, encoding="utf-8-sig")
(classif[classif['decisao'] == 'EXCLUIR'][['ticker', 'motivo', 'anos_penny', 'anos_situacao', 'codigos_linhagem']]
    .to_csv(PASTA_SAIDA / "tickers_excluidos_integridade.csv", index=False, encoding="utf-8-sig"))
(classif[classif['decisao'] == 'MANTER'][['ticker']]
    .to_csv(PASTA_SAIDA / "tickers_finais_v2.csv", index=False, encoding="utf-8-sig"))

print("=== RESUMO DA CLASSIFICAÇÃO ===")
for d in ['MANTER', 'EXCLUIR', 'SEM DADOS']:
    print(f"  {d}: {(classif['decisao'] == d).sum()}")

print("\n--- EXCLUÍDOS ---")
for _, r in classif[classif['decisao'] == 'EXCLUIR'].iterrows():
    print(f"  {r['ticker']:8s} | {r['motivo']}")

sem = classif[classif['decisao'] == 'SEM DADOS']['ticker'].tolist()
if sem:
    print("\n--- SEM DADOS no COTAHIST (revisar manualmente; possível renomeação) ---")
    print(" ", ", ".join(sem))

print(f"\nArquivos gravados em: {PASTA_SAIDA}")
print("  - classificacao_universo.csv")
print("  - tickers_excluidos_integridade.csv")
print("  - tickers_finais_v2.csv")

=== RESUMO DA CLASSIFICAÇÃO ===
  MANTER: 93
  EXCLUIR: 28
  SEM DADOS: 1

--- EXCLUÍDOS ---
  AMAR3    | penny: PRECO_UNIT < R$1 em AMAR3 [2023, 2024, 2025]
  BEES3    | penny: PRECO_UNIT < R$1 em BEES3 [2012, 2013, 2014, 2015]
  ENEV3    | penny: PRECO_UNIT < R$1 em ENEV3 [2014, 2015, 2016]; situacao especial em ENEV3 [2014, 2015, 2016]
  ETER3    | penny: PRECO_UNIT < R$1 em ETER3 [2017, 2018]; situacao especial em ETER3 [2018, 2019, 2020, 2021, 2022, 2023, 2024]
  FHER3    | situacao especial em FHER3 [2019, 2020, 2021, 2022]
  FICT3    | penny: PRECO_UNIT < R$1 em ATOM3 [2015, 2016] (via predecessor); situacao especial em ATOM3 [2015, 2016, 2017] (via predecessor)
  GOAU3    | penny: PRECO_UNIT < R$1 em GOAU3 [2016]
  GOAU4    | penny: PRECO_UNIT < R$1 em GOAU4 [2016]
  GOLL54   | penny: PRECO_UNIT < R$1 em GOLL54 [2025], GOLL4 [2024, 2025] (via predecessor)
  HAGA4    | penny: PRECO_UNIT < R$1 em HAGA4 [2015, 2016, 2023]; situacao especial em HAGA4 [2010, 2011, 2012]
  HBOR3    |